[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/sessions/3/notebook.ipynb)



# Session 3: From Counting to Learning - Cross-Entropy and Neural Networks

**Read this alongside `lesson.md`** — this notebook contains exercises referenced in the lesson.

In Session 2, we built bigram models by counting transitions. This works perfectly when V is small (28 characters), but hits the **V² wall** when vocabularies grow large. Session 3 introduces the solution: **embeddings** and **neural networks**.

But first, we need to formalize what we mean by "loss" and "learning" — which brings us to **cross-entropy** and **KL divergence**.

## Part 0: Setup

Make sure you have the course package installed:

In [ ]:
# Install the course package
%pip install -q git+https://github.com/BaytAlhikmah/hands-on-llms-for-swes.git#subdirectory=pkg

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from alhikmah_llms import Session3

print("Setup complete!")

---

## Part 1: Cross-Entropy - Formalizing "Loss"

_Corresponds to lesson sections 1-3_

In Session 2, we called it "average surprise." In machine learning, it's called **cross-entropy**.

### Exercise 1: The Core Intuition

_Lesson section 1: What is cross-entropy?_

Cross-entropy H(P,Q) measures how well a predicted distribution Q matches the true distribution P.

**Example:** After 'q', what character comes next?
- True distribution P: 'u' appears 70% of the time, 'a' 20%, 'e' 10%
- Bad model Q: predicts uniformly (33% each)
- Good model Q: predicts close to true (65%, 25%, 10%)

**Lower cross-entropy = better model**

In [ ]:
# Visualize: true distribution vs two models
Session3.plot_cross_entropy_intuition()
plt.show()

print("🎯 Key Insight:")
print("   The model with predictions closer to the true distribution")
print("   has LOWER cross-entropy = BETTER performance!")

### Exercise 2: Where Does the Loss Come From?

_Lesson section 2: Loss breakdown_

Cross-entropy is computed as:

```
H(P,Q) = -∑ P(x) log Q(x)
```

Each outcome contributes `-P(x) log Q(x)` to the total loss.

**Question:** Which outcomes matter most?

In [ ]:
# Breakdown: see which outcomes contribute most
Session3.plot_surprise_breakdown()
plt.show()

print("\n🎯 Key Observation:")
print("   Outcomes with HIGH TRUE PROBABILITY (large P) contribute most!")
print("   Getting common cases right matters more than rare ones.")

### Exercise 3: The Mathematical Formula

_Lesson section 2: Computing cross-entropy_

Let's implement cross-entropy from scratch and verify it matches our intuition.

In [ ]:
def cross_entropy(P: list[float], Q: list[float]) -> float:
    """Compute H(P,Q) = -∑ P(x) log Q(x)
    
    Args:
        P: True probability distribution
        Q: Predicted probability distribution
    
    Returns:
        Cross-entropy in nats (natural log units)
    """
    return -sum(p * np.log(q) if q > 0 else 0 for p, q in zip(P, Q))


# Test cases
P_true = [0.7, 0.2, 0.1]

# Perfect prediction
Q_perfect = [0.7, 0.2, 0.1]
print(f"Perfect model: H(P,Q) = {cross_entropy(P_true, Q_perfect):.4f}")

# Good prediction
Q_good = [0.65, 0.25, 0.1]
print(f"Good model:    H(P,Q) = {cross_entropy(P_true, Q_good):.4f}")

# Bad prediction (uniform)
Q_bad = [0.33, 0.33, 0.34]
print(f"Bad model:     H(P,Q) = {cross_entropy(P_true, Q_bad):.4f}")

# Terrible prediction (backwards!)
Q_terrible = [0.1, 0.2, 0.7]
print(f"Terrible model: H(P,Q) = {cross_entropy(P_true, Q_terrible):.4f}")

print("\nNotice: Loss increases as predicted distribution gets worse!")

### Exercise 4: Bits vs Nats

_Lesson section 3: Units of information_

**Important:** In Session 2, we used **log₂** and measured entropy in **bits**. In machine learning (and from now on), we use **natural log (ln)** and measure in **nats**.

Why?
- PyTorch's `nn.CrossEntropyLoss()` uses natural log
- Derivatives are cleaner: d/dx(ln x) = 1/x
- Standard in ML papers and frameworks

**Conversion:** `bits = nats / ln(2) ≈ nats * 1.443`

In [ ]:
# Compare bits vs nats
P = [0.7, 0.2, 0.1]
Q = [0.65, 0.25, 0.1]

# In nats (natural log)
ce_nats = cross_entropy(P, Q)

# In bits (log2)
ce_bits = -sum(p * math.log2(q) if q > 0 else 0 for p, q in zip(P, Q))

print(f"Cross-entropy in NATS: {ce_nats:.4f}")
print(f"Cross-entropy in BITS: {ce_bits:.4f}")
print(f"Ratio: {ce_bits / ce_nats:.4f} ≈ 1 / ln(2) ≈ 1.443")

print("\nFrom now on, we use NATS (natural log) to match ML conventions.")

---

## Part 2: The Key Decomposition - Entropy + KL Divergence

_Corresponds to lesson sections 4-5_

Here's the most important equation in this session:

```
H(P,Q) = H(P) + KL(P||Q)
```

Where:
- **H(P,Q)** = cross-entropy (what we minimize in training)
- **H(P)** = entropy of true distribution (constant, can't change)
- **KL(P||Q)** = KL divergence (measures how Q differs from P)

### Exercise 5: Visualizing the Decomposition

_Lesson section 4: The decomposition_

**Key insight:** When training a model, we can't change H(P) (it's a property of the data). We're really just minimizing KL(P||Q)!

In [ ]:
Session3.plot_entropy_decomposition()
plt.show()

print("\n🎯 Critical Insight:")
print("   Training doesn't reduce H(P) — that's fixed by the data!")
print("   Training minimizes KL(P||Q) — making Q match P.")
print("   Minimum loss = H(P) when Q = P (KL = 0)")

### Exercise 6: Computing KL Divergence

_Lesson section 5: What is KL divergence?_

KL divergence (Kullback-Leibler divergence) measures how one distribution differs from another:

```
KL(P||Q) = ∑ P(x) log(P(x) / Q(x))
         = ∑ P(x) log P(x) - ∑ P(x) log Q(x)
         = -H(P) + H(P,Q)
```

Properties:
- Always non-negative: KL(P||Q) ≥ 0
- Zero only when P = Q
- **NOT symmetric:** KL(P||Q) ≠ KL(Q||P)

In [ ]:
def entropy(P: list[float]) -> float:
    """Compute H(P) = -∑ P(x) log P(x)"""
    return -sum(p * np.log(p) for p in P if p > 0)

def kl_divergence(P: list[float], Q: list[float]) -> float:
    """Compute KL(P||Q) = ∑ P(x) log(P(x) / Q(x))"""
    return sum(p * np.log(p / q) if p > 0 and q > 0 else 0 
               for p, q in zip(P, Q))

# Verify the decomposition
P = [0.7, 0.2, 0.1]
Q = [0.5, 0.3, 0.2]

h_p = entropy(P)
h_pq = cross_entropy(P, Q)
kl_pq = kl_divergence(P, Q)

print("Verify H(P,Q) = H(P) + KL(P||Q):")
print(f"  H(P)      = {h_p:.6f}")
print(f"  KL(P||Q)  = {kl_pq:.6f}")
print(f"  H(P) + KL = {h_p + kl_pq:.6f}")
print(f"  H(P,Q)    = {h_pq:.6f}")
print(f"  Match? {abs(h_pq - (h_p + kl_pq)) < 1e-10}")

### Exercise 7: KL Divergence is NOT Symmetric

_Lesson section 5: Asymmetry matters_

This is important! KL(P||Q) ≠ KL(Q||P), and the difference matters for how models learn.

- **Forward KL:** KL(P||Q) — penalizes heavily when Q assigns low probability where P is high ("mode-seeking")
- **Reverse KL:** KL(Q||P) — penalizes heavily when Q assigns high probability where P is low ("mode-covering")

In [ ]:
Session3.plot_kl_asymmetry()
plt.show()

print("\n🎯 Key Point:")
print("   In ML, we minimize KL(P||Q) where P = data, Q = model")
print("   This is 'forward KL' — model must cover all modes of data!")

In [ ]:
# Numerical example
P = [0.8, 0.15, 0.05]  # peaked distribution
Q = [0.4, 0.3, 0.3]     # flat distribution

kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)

print(f"KL(P||Q) = {kl_pq:.4f}")
print(f"KL(Q||P) = {kl_qp:.4f}")
print(f"Difference: {abs(kl_pq - kl_qp):.4f}")
print("\nThey're NOT equal!")

---

## Part 3: Training = Minimizing Cross-Entropy

_Corresponds to lesson sections 6-7_

Now we understand what "training" really means: **adjust Q to minimize H(P,Q)**.

### Exercise 8: Watch a Model Converge

_Lesson section 6: Training visualization_

As training progresses, the predicted distribution Q moves closer to the true distribution P, and cross-entropy decreases.

In [ ]:
Session3.plot_training_trajectory()
plt.show()

print("\n🎯 What's Happening:")
print("   The blue bars (model) gradually match the green bars (truth)")
print("   As they align, cross-entropy decreases toward H(P)")
print("   This IS what happens inside neural network training!")

### Exercise 9: The Loss Surface

_Lesson section 7: Loss landscape_

For a 3-outcome distribution, we can visualize the entire loss surface as we vary Q's parameters.

**Key observation:** There's a unique minimum at Q = P.

In [ ]:
Session3.plot_cross_entropy_surface(
    true_probs=[0.7, 0.2, 0.1]
)
plt.show()

print("\n🎯 Observations:")
print("   - Red star (Q = P) is the global minimum")
print("   - Loss increases as Q moves away from P")
print("   - Gradient descent would roll downhill to the minimum")
print("   - (In neural nets, we optimize weights, not Q directly!)")

---

## Part 4: Back to Bigrams - Cross-Entropy in Practice

_Corresponds to lesson sections 8-9_

Let's connect this to the bigram model from Session 2. After 'q', what comes next?

### Exercise 10: Real Bigram Example

_Lesson section 8: Bigram cross-entropy_

Watch cross-entropy decrease as a bigram model learns.

In [ ]:
Session3.plot_bigram_cross_entropy_example()
plt.show()

print("\n🎯 What You're Seeing:")
print("   - Training data: 'qu' appears 850 times, others are rare")
print("   - Early training: model is nearly uniform (bad!)")
print("   - Late training: model learns to predict 'u' after 'q' (good!)")
print("   - Cross-entropy drops from ~1.6 to ~0.06 nats")

### Exercise 11: Loss Curves

_Lesson section 9: Monitoring training_

In practice, we plot loss over training epochs to monitor progress.

In [ ]:
# Generate example loss curves
Session3.plot_loss_curve()
plt.show()

print("\n🎯 What to Watch For:")
print("   - Training loss should decrease smoothly")
print("   - Validation loss should track training loss")
print("   - If val >> train → overfitting")
print("   - If both plateau → model converged (or stuck in local minimum)")

---

## Part 5: From Counting to Neural Networks

_Corresponds to lesson sections 10-12_

So far, everything we've done could be implemented with **counting** (like Session 2). But we hit the V² wall:

- 28 characters: 784 parameters (manageable)
- 50K tokens: 2.5 billion parameters (impossible to count, expensive to store)

The solution: **neural networks with embeddings**.

### Exercise 12: Custom Cross-Entropy Examples

_Lesson section 10: Experiment yourself_

Try different distributions and see how cross-entropy changes.

In [ ]:
# TODO: Experiment with different distributions

# Example 1: Very confident but wrong
P = [0.9, 0.08, 0.02]
Q = [0.1, 0.1, 0.8]  # model is confident but backwards!

print("Example 1: Confident but wrong")
print(f"  True P = {P}")
print(f"  Model Q = {Q}")
print(f"  Cross-entropy = {cross_entropy(P, Q):.4f}")
print(f"  (Very high! Model is confidently wrong.)")
print()

# Example 2: Uncertain but safe
Q_uniform = [0.33, 0.33, 0.34]
print("Example 2: Uniform (uncertain)")
print(f"  Model Q = {Q_uniform}")
print(f"  Cross-entropy = {cross_entropy(P, Q_uniform):.4f}")
print(f"  (Lower than Example 1! Being uncertain is better than being wrong.)")
print()

# Your turn: create more examples!
# What happens when Q is very close to P?
# What happens when Q completely misses a high-probability outcome?

### Exercise 13: Visualize Your Own Examples

_Lesson section 11: Build intuition_

Use the visualization functions with your own distributions.

In [ ]:
# Create your own example
my_true = [0.6, 0.3, 0.1]
my_pred_bad = [0.2, 0.4, 0.4]
my_pred_good = [0.58, 0.32, 0.1]

Session3.plot_cross_entropy_intuition(
    true_probs=my_true,
    predicted_bad=my_pred_bad,
    predicted_good=my_pred_good,
    labels=['A', 'B', 'C']
)
plt.show()

---

## Summary and What's Next

### What We Learned

1. **Cross-entropy** H(P,Q) measures how well predicted distribution Q matches true distribution P
2. **Decomposition**: H(P,Q) = H(P) + KL(P||Q)
   - H(P) is constant (property of data)
   - Training minimizes KL(P||Q)
3. **KL divergence** is NOT symmetric: KL(P||Q) ≠ KL(Q||P)
4. **Training** = adjusting model parameters to minimize cross-entropy
5. We use **natural log (nats)** in ML, not log₂ (bits)

### The Problem We Face

Session 2's counting approach works for small vocabularies but hits the **V² wall**:
- 128K vocabulary → 16 billion parameters just for a bigram matrix!
- Can't count every possible sequence in large vocabularies
- Need better generalization

### What's Next (Session 4+)

**Neural networks with embeddings** solve both problems:
1. **Compression**: Represent vocabulary with V×d matrix + small network (not V×V)
2. **Generalization**: Learn patterns that work for unseen sequences

Coming up:
- Embeddings: representing discrete tokens as continuous vectors
- Neural bigram model: tiny network learns same patterns as counting
- Backpropagation: how networks actually learn
- Extending context: from bigrams to trigrams and beyond
- Eventually: attention and transformers

---

## Exercises for Practice

1. **Derive the decomposition**: Prove that H(P,Q) = H(P) + KL(P||Q) using the definitions
2. **Find worst case**: For P = [0.7, 0.2, 0.1], what Q maximizes cross-entropy?
3. **Asymmetry exploration**: Create distributions where KL(P||Q) and KL(Q||P) differ dramatically
4. **Real data**: Load the Session 2 bigram counts, compute cross-entropy for different models
5. **Temperature**: If Q = softmax(logits / T), how does temperature affect cross-entropy?